# Agent Observability

**Level:** Advanced · **Time:** 60 min

Agent observability answers the critical production question: **Why did the system do that?**

Unlike traditional software where you can step through predictable `if/else` statements, Agentic workflows are stochastic (random). They rely on LLMs reasoning over dynamic context to choose which tools to execute. When an agent fails, you cannot just look at a stack trace. You need to look at the **OpenTelemetry Trace Hierarchy**.

In this notebook, we will demonstrate:
1. Planning Observability with Correlation IDs.
2. Understanding the raw Telemetry Logs (Spans).
3. Instrumenting the OTel Trace Hierarchy using **AgentOps**.
4. Capturing advanced KPIs (Cost, Latency, and Evals).
5. Debugging common Agent failures like infinite reasoning loops.

> **Note:** The code blocks use the `agentops` and `opentelemetry` SDKs. To allow this notebook to run without active API keys or a cloud dashboard account, the trace data outputs are simulated in the cells below.

---
## Pattern 1: Planning Observability & Correlation IDs

In complex multi-agent systems, a user request might hit an API gateway, get routed to a Supervisor Agent, which then triggers a background job handled by a Worker Agent. 

To ensure all these logs appear in the same trace dashboard, you must pass **Correlation IDs** (W3C Trace Context) between your services.

In [1]:
import json
from opentelemetry import trace
from opentelemetry.trace.propagation.tracecontext import TraceContextTextMapPropagator

# 1. Supervisor Agent creates a trace context
tracer = trace.get_tracer(__name__)
with tracer.start_as_current_span("SupervisorAgent") as span:
    
    # Extract the trace headers to send to the Worker Agent over HTTP/Kafka
    headers = {}
    TraceContextTextMapPropagator().inject(headers)
    print("Injected Trace Headers to send to Worker:")
    print(json.dumps(headers, indent=2))
    
# 2. Worker Agent receives the headers and continues the trace
print("\n[Worker Agent] Resuming trace from headers...")
ctx = TraceContextTextMapPropagator().extract(carrier=headers)
with tracer.start_as_current_span("WorkerAgent_Task", context=ctx):
    print("Worker is now executing within the same Trace ID!")

Injected Trace Headers to send to Worker:
{}

[Worker Agent] Resuming trace from headers...
Worker is now executing within the same Trace ID!


---
## Pattern 2: Instrumenting the OTel Trace Hierarchy

![Observability Hierarchy](../../../assets/observability_trace_hierarchy.svg)

A proper observability trace organizes an agent's execution into a tree:
`Session -> Agent -> LLM Spans & Tool Spans`.

Using decorators from an observability SDK like **AgentOps**, we can automatically wrap our Python functions so that every time they execute, they emit a "Span" to the dashboard containing the arguments, returns, latency, and token counts.

In [2]:
import agentops
from agentops import tool

# 1. Initialize the Observability Session
agentops.init(api_key="mock_key", auto_start_session=False)
session = agentops.start_session(tags=["incident-response", "eu-checkout"])

# 2. Instrument Tools with @tool
@tool(name="Tool Span")
def query_datadog(metric: str) -> str:
    print(f"[Tool] Querying Datadog for: {metric}")
    return "Error Rate: 15%"

@tool(name="Tool Span")
def fetch_logs(service: str) -> str:
    print(f"[Tool] Fetching logs for: {service}")
    return "Log: 504 Gateway Timeout on deploy-842"

# 3. Instrument the Agent's Core Logic
@tool(name="Agent Workflow")
def run_investigation_agent(incident_id: str):
    print(f"[Agent] Starting investigation for {incident_id}")
    
    # In reality, an LLM would choose to call these tools. We simulate that here.
    query_datadog("eu_checkout_success_rate")
    fetch_logs("checkout-api")
    
    return "Root Cause: deploy-842"

# Execute
result = run_investigation_agent("INC-104")
agentops.end_session("Success")

🖇 AgentOps: start_session() is deprecated and will be removed in v4 in the future. Use agentops.start_trace() instead.


🖇 AgentOps: Session Replay for session trace: https://app.agentops.ai/sessions?trace_id=fcb7932ac1bc656fca2d0dc7e331672f


🖇 AgentOps: end_session() is deprecated and will be removed in v4 in the future. Use agentops.end_trace() instead.


[Agent] Starting investigation for INC-104
[Tool] Querying Datadog for: eu_checkout_success_rate
[Tool] Fetching logs for: checkout-api


🖇 AgentOps: [agentops.InternalSpanProcessor] Error uploading logfile: Upload failed: 401


🖇 AgentOps: Session Replay for session.session trace: https://app.agentops.ai/sessions?trace_id=fcb7932ac1bc656fca2d0dc7e331672f


---
## Pattern 3: The Raw Telemetry Log (Span)

When an LLM or tool span completes, what actually gets sent to Datadog or AgentOps? It is a structured JSON payload conforming to Semantic Conventions.

Notice how it captures exact token counts and the raw prompt content. **Warning:** If your user passes PII (like a Social Security Number) in the prompt, it will be logged here unless you redact it first!

In [3]:
raw_span = {
  "trace_id": "4bf92f3577b34da6a3ce929d0e0e4736",
  "span_id": "00f067aa0ba902b7",
  "name": "chat.completions.create",
  "attributes": {
    "llm.system": "openai",
    "llm.model": "gpt-4o",
    "llm.request.temperature": 0.2,
    "llm.usage.prompt_tokens": 4092,
    "llm.usage.completion_tokens": 128,
    "llm.usage.total_tokens": 4220,
    "llm.prompts.0.content": "You are a database admin. Generate SQL to find... [REDACTED PII]",
    "llm.completions.0.content": "SELECT * FROM users WHERE status = 'error'"
  },
  "start_time": "2026-08-13T10:00:00.000Z",
  "end_time": "2026-08-13T10:00:02.150Z" # TTFT (Time To First Token) and Latency are derived from here
}

print("Raw OTel LLM Span Output:")
print(json.dumps(raw_span, indent=2))

Raw OTel LLM Span Output:
{
  "trace_id": "4bf92f3577b34da6a3ce929d0e0e4736",
  "span_id": "00f067aa0ba902b7",
  "name": "chat.completions.create",
  "attributes": {
    "llm.system": "openai",
    "llm.model": "gpt-4o",
    "llm.request.temperature": 0.2,
    "llm.usage.prompt_tokens": 4092,
    "llm.usage.completion_tokens": 128,
    "llm.usage.total_tokens": 4220,
    "llm.prompts.0.content": "You are a database admin. Generate SQL to find... [REDACTED PII]",
    "llm.completions.0.content": "SELECT * FROM users WHERE status = 'error'"
  },
  "start_time": "2026-08-13T10:00:00.000Z",
  "end_time": "2026-08-13T10:00:02.150Z"
}


---
## Pattern 4: Capturing Advanced KPIs (Cost, Latency, & Evals)

![KPI Dashboard](../../../assets/observability_kpi_dashboard.svg)

By instrumenting your LLM calls, the SDK automatically captures the exact number of tokens used. You can also run asynchronous **Evals** (LLM-as-a-judge) to grade the groundedness or accuracy of the trace.

In [4]:
# In AgentOps v0.4+, LLM calls are tracked automatically via SDK wrappers!
# We simulate the automatically extracted metadata below:

def simulate_llm_call_and_eval():
    # 1. Simulate the automatically recorded LLM Call (Cost & Tokens)
    event = {
        "model": "gpt-4o",
        "prompt_tokens": 15420,  # Massive context!
        "completion_tokens": 15,
        "cost": 0.077            # Calculated automatically by SDKs
    }
    print(f"Recorded LLM Event: {event['model']}, Cost: ${event['cost']}, Total Tokens: {event['prompt_tokens'] + event['completion_tokens']}")
    
    # 2. Behavioral KPI: LLM-as-a-judge Eval
    # We evaluate if the LLM hallucinated, then tag the session with the result.
    eval_score = 0.95 # Simulated evaluation score
    print(f"Eval: Groundedness Score = {eval_score}. Task is highly grounded in RAG context.")
    
simulate_llm_call_and_eval()


Recorded LLM Event: gpt-4o, Cost: $0.077, Total Tokens: 15435
Eval: Groundedness Score = 0.95. Task is highly grounded in RAG context.


---
## Pattern 5: Debugging Failures (The Infinite Loop)

The most common autonomous failure mode is the **Infinite Reasoning Loop**. 
This happens when an LLM chooses a tool, provides invalid arguments, receives an error, and blindly retries the exact same invalid arguments.

Without observability, the request eventually times out. With observability, you can look at the trace and immediately spot the loop and the resulting token spike.

In [5]:
@tool(name="Tool Span")
def broken_sql_query(query: str):
    # This tool expects valid SQL, but the LLM keeps passing natural language.
    return "Syntax Error: near 'show me'"

@tool(name="Agent Workflow")
def runaway_agent():
    print("[Agent] Trying to find the error...")
    
    # Simulating the LLM getting stuck in a loop
    for i in range(1, 6):
        print(f"  Iteration {i}: LLM calls broken_sql_query()")
        result = broken_sql_query("show me the errors for today")
        
    print("[System] CRASH: Maximum Token Limit Exceeded")
    
runaway_agent()

[Agent] Trying to find the error...
  Iteration 1: LLM calls broken_sql_query()
  Iteration 2: LLM calls broken_sql_query()
  Iteration 3: LLM calls broken_sql_query()
  Iteration 4: LLM calls broken_sql_query()
  Iteration 5: LLM calls broken_sql_query()
[System] CRASH: Maximum Token Limit Exceeded


---
## Technology Review: State of the Art

When choosing an observability stack for your agents, consider:

1. **Native LLM Platforms (AgentOps, LangSmith, Arize Phoenix)**: Built specifically for agents. They offer beautiful UI trace replays, built-in LLM-as-a-judge Evals, and cost tracking out of the box. Best for AI Engineering teams.
2. **Traditional APMs (Datadog, New Relic)**: Adopting OpenTelemetry semantics rapidly. Best if your Platform Engineering team insists on keeping all agent logs in the exact same dashboard as your Kubernetes clusters and Postgres databases.